# 🥋 Judo Throws Classification with X3D
**DATA.ML.330 Media Analysis — Group 4**

**Project:** Deep Learning-Based Judo Throws Classification

**Authors:** Alexander Gradov & Oluwaseun Akangbe (Samuel)

**Model:** X3D-M (Meta Research) pre-trained on Kinetics-400

**Dataset:** [adenhaus/judo_throws](https://huggingface.co/datasets/adenhaus/judo_throws) — 805 videos, 4 throw classes

---

## How to Use This Notebook
1. **Runtime → Change runtime type → T4 GPU** (free tier)
2. Run all cells in order (Runtime → Run all)
3. Training takes ~15-30 min on T4 GPU
4. Results (model, plots, metrics) are saved and can be downloaded

## Step 0: Verify GPU is Available
Make sure you selected **T4 GPU** under Runtime → Change runtime type.

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    total_mem = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', None)
    if total_mem:
        print(f"GPU Memory: {total_mem / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


AttributeError: 'torch._C._CudaDeviceProperties' object has no attribute 'total_mem'

## Step 1: Install Dependencies
Install PyTorchVideo (X3D models), PyAV (video decoding), and other packages.

In [5]:
!pip install -q pytorchvideo av huggingface_hub datasets scikit-learn matplotlib tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.7/132.7 kB 6.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 22.5 MB/s eta 0:00:00


## Step 2: Clone the GitHub Repository
This pulls our project code from GitHub. **Replace the URL below with your actual repo URL.**

In [ ]:
import os

# ========================================================================
# OPTION A: Public repo — just use the URL directly
# OPTION B: Private repo — use a Personal Access Token (PAT)
#   1. Go to GitHub → Settings → Developer settings → Personal access tokens
#   2. Generate a token with 'repo' scope
#   3. Paste it below
# ========================================================================
REPO_URL = "https://github.com/safo-124/media-analysis.git"
PROJECT_DIR = "/content/media-analysis"

# If your repo is PRIVATE, uncomment and fill in your token:
# GITHUB_TOKEN = "ghp_YOUR_TOKEN_HERE"
# REPO_URL = f"https://{GITHUB_TOKEN}@github.com/safo-124/media-analysis.git"

if os.path.exists(PROJECT_DIR):
    print("Repo already cloned — pulling latest changes...")
    !cd {PROJECT_DIR} && git pull
else:
    !git clone {REPO_URL} {PROJECT_DIR}

if not os.path.exists(PROJECT_DIR):
    raise RuntimeError(
        "❌ Clone failed! If your repo is private, uncomment the GITHUB_TOKEN lines above "
        "and paste your Personal Access Token."
    )

os.chdir(PROJECT_DIR)
print(f"✅ Working directory: {os.getcwd()}")
!ls -la

Cloning into '/content/media-analysis'...
fatal: could not read Username for 'https://github.com': No such device or address
Cloned to /content/media-analysis


FileNotFoundError: [Errno 2] No such file or directory: '/content/media-analysis'

## Step 3: Download the Judo Throws Dataset
Download directly from Hugging Face — this is faster than uploading from your PC.

**Dataset details:**
- Source: [adenhaus/judo_throws](https://huggingface.co/datasets/adenhaus/judo_throws)
- 4 classes: Ippon Seoi Nage, O Goshi, Osoto Gari, Uchi Mata
- 602 train / 102 val / 101 test videos
- ~470 MB total

In [7]:
from huggingface_hub import snapshot_download

DATA_DIR = os.path.join(PROJECT_DIR, "judo_throws_dataset")

if os.path.exists(DATA_DIR):
    print(f"Dataset already exists at {DATA_DIR}")
else:
    print("Downloading dataset from Hugging Face...")
    snapshot_download(
        repo_id="adenhaus/judo_throws",
        repo_type="dataset",
        local_dir=DATA_DIR
    )
    print(f"Downloaded to {DATA_DIR}")

# Verify dataset
for split in ['train', 'val', 'test']:
    sp = os.path.join(DATA_DIR, split)
    classes = [c for c in sorted(os.listdir(sp)) if os.path.isdir(os.path.join(sp, c))]
    total = sum(len(os.listdir(os.path.join(sp, c))) for c in classes)
    print(f"  {split.upper()}: {total} videos across {len(classes)} classes")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 810 files:   0%|          | 0/810 [00:00<?, ?it/s]

Downloaded to /content/media-analysis/judo_throws_dataset
  TRAIN: 602 videos across 4 classes
  VAL: 102 videos across 4 classes
  TEST: 101 videos across 4 classes


## Step 4: Configure for Colab
Override the config paths to point to Colab's file system instead of the Windows paths.

**Why?** The config.py has Windows paths (`e:\media analysis\...`) that don't exist on Colab.
We override them at runtime so the same codebase works on both Windows and Colab.

In [ ]:
import sys
import os

# Ensure PROJECT_DIR is set (in case cells were run out of order)
PROJECT_DIR = "/content/media-analysis"
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

# Override config paths for Colab environment
import configs.config as config

config.DATA_DIR = os.path.join(PROJECT_DIR, "judo_throws_dataset")
config.OUTPUT_DIR = os.path.join(PROJECT_DIR, "outputs")
os.makedirs(config.OUTPUT_DIR, exist_ok=True)

# Use GPU on Colab
config.DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Colab can handle more workers and bigger batches with T4 GPU (16GB VRAM)
config.BATCH_SIZE = 8       # Increase from 4 → 8 (T4 has 16GB VRAM)
config.NUM_WORKERS = 2      # Colab supports multiprocessing (unlike Windows)

print("Colab Configuration:")
print(f"  Device:     {config.DEVICE}")
print(f"  Data dir:   {config.DATA_DIR}")
print(f"  Output dir: {config.OUTPUT_DIR}")
print(f"  Batch size: {config.BATCH_SIZE}")
print(f"  Workers:    {config.NUM_WORKERS}")
print(f"  Epochs:     {config.NUM_EPOCHS}")
print(f"  LR:         {config.LEARNING_RATE}")

ModuleNotFoundError: No module named 'configs'

## Step 5: Load the Dataset
This scans the video folders, builds the dataset objects, and wraps them in DataLoaders.

**What happens:**
1. Each split folder (train/val/test) is scanned for `.mp4` files
2. Subfolder names become class labels (e.g., `ippon_seoi_nage` → class 0)
3. Training data gets random crop + flip augmentation
4. Val/test data gets deterministic center crop
5. All frames are normalized with Kinetics mean/std

In [ ]:
from src.dataset import get_dataloaders, CLASS_NAMES

dataloaders = get_dataloaders()

# Verify a batch loads correctly
videos, labels = next(iter(dataloaders['train']))
print(f"\n✅ Data loading successful!")
print(f"  Batch shape: {videos.shape}  →  (batch, channels, frames, height, width)")
print(f"  Labels: {labels.tolist()}")
print(f"  Classes: {[CLASS_NAMES[l] for l in labels.tolist()]}")

## Step 6: Visualize Sample Videos
Let's look at what the model will see — sampled frames from each throw class.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from src.dataset import decode_video

fig, axes = plt.subplots(4, 8, figsize=(20, 10))
fig.suptitle('Sample Frames from Each Judo Throw Class', fontsize=16)

for class_idx, class_name in enumerate(CLASS_NAMES):
    # Get first video of this class from training set
    class_dir = os.path.join(config.DATA_DIR, 'train', class_name)
    videos_list = [f for f in sorted(os.listdir(class_dir))
                   if f.endswith('.mp4') and not f.startswith('.')]
    video_path = os.path.join(class_dir, videos_list[0])

    # Decode and sample 8 frames (every other frame from 16)
    frames = decode_video(video_path, num_frames=16)
    display_frames = frames[::2]  # Show 8 of 16 frames

    for frame_idx in range(8):
        ax = axes[class_idx, frame_idx]
        ax.imshow(display_frames[frame_idx])
        ax.axis('off')
        if frame_idx == 0:
            ax.set_ylabel(class_name.replace('_', '\n'), fontsize=10, rotation=0,
                         labelpad=80, va='center')

plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUT_DIR, 'sample_frames.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Sample frames saved to outputs/sample_frames.png")

## Step 7: Build the X3D Model
Load X3D-M pre-trained on Kinetics-400 and replace the classification head.

**What is X3D?**
- Efficient video recognition model from Meta Research
- Expands a 2D image model along 6 axes (temporal, spatial, width, depth, etc.)
- X3D-M: 3M params, 4.7 GFLOPs — much lighter than SlowFast or Video Swin

**Transfer Learning:**
- Backbone: pre-trained features (edges → body parts → actions)
- Head: new `Linear(2048 → 4)` layer for our 4 judo throw classes
- We fine-tune everything with a small learning rate (1e-4)

In [ ]:
from src.model import build_model

model = build_model()

# Verify forward pass on GPU
dummy = torch.randn(1, 3, 16, 224, 224).to(config.DEVICE)
with torch.no_grad():
    output = model(dummy)
print(f"\n✅ Model forward pass successful!")
print(f"  Input:  {dummy.shape}")
print(f"  Output: {output.shape} → {config.NUM_CLASSES} class logits")

## Step 8: Train the Model 🚀
This is the main training loop. On a T4 GPU, expect:
- ~1-2 minutes per epoch
- ~25-50 minutes total (with early stopping)

**What happens each epoch:**
1. **Train**: Forward pass → Loss → Backward pass → Update weights
2. **Validate**: Forward pass only → measure generalization
3. **Schedule**: Reduce LR if val loss plateaus
4. **Checkpoint**: Save model if val accuracy improves
5. **Early stop**: If no improvement for 10 epochs, stop

In [ ]:
from src.train import train

model, history = train(model, dataloaders)

print("\n✅ Training complete!")

## Step 9: Evaluate on Test Set
Run the trained model on the held-out test set and generate all evaluation outputs:
- Overall test accuracy
- Confusion matrix (which throws get confused?)
- Classification report (precision, recall, F1 per class)
- Training curves (loss & accuracy over epochs)
- Per-class accuracy bar chart

In [ ]:
from src.evaluate import full_evaluation

results = full_evaluation(model, dataloaders['test'], history=history)

print(f"\n🏆 FINAL TEST ACCURACY: {results['test_accuracy']:.1%}")

## Step 10: Download Results
Download the trained model and all outputs to your local machine.

In [ ]:
# Zip all outputs for easy download
import shutil

output_zip = os.path.join(PROJECT_DIR, "judo_results")
shutil.make_archive(output_zip, 'zip', config.OUTPUT_DIR)
print(f"Results zipped to: {output_zip}.zip")

# In Colab, this triggers a download dialog
try:
    from google.colab import files
    files.download(f"{output_zip}.zip")
    print("Download started!")
except ImportError:
    print("Not running in Colab — find the zip at:", f"{output_zip}.zip")

## Step 11: Save Model to Google Drive (Optional)
Mount Google Drive to persist the trained model across Colab sessions.

In [ ]:
# Uncomment the lines below to save to Google Drive

# from google.colab import drive
# drive.mount('/content/drive')
#
# drive_output = '/content/drive/MyDrive/judo_throws_results'
# os.makedirs(drive_output, exist_ok=True)
#
# import shutil
# for f in os.listdir(config.OUTPUT_DIR):
#     src = os.path.join(config.OUTPUT_DIR, f)
#     dst = os.path.join(drive_output, f)
#     shutil.copy2(src, dst)
#     print(f"Copied: {f}")
#
# print(f"\nAll results saved to Google Drive: {drive_output}")

---
## Summary

| Component | Detail |
|---|---|
| **Model** | X3D-M (Meta Research) |
| **Pre-training** | Kinetics-400 (400 action classes) |
| **Dataset** | adenhaus/judo_throws (805 videos, 4 classes) |
| **Classes** | Ippon Seoi Nage, O Goshi, Osoto Gari, Uchi Mata |
| **Input** | 16 frames × 224×224, Kinetics-normalized |
| **Optimizer** | Adam (lr=1e-4, weight_decay=1e-4) |
| **Loss** | CrossEntropyLoss |
| **Scheduler** | ReduceLROnPlateau (patience=5, factor=0.1) |
| **Early Stopping** | Patience=10 epochs |

### Reference
Feichtenhofer, C. (2020). *X3D: Expanding Architectures for Efficient Video Recognition.*
CVPR 2020. [Paper](https://ai.meta.com/research/publications/x3d-expanding-architectures-for-efficient-video-recognition/)